In [1]:
import math
import random

# Activation functions

Les fonctions d'activation introduisent de la **non-linéarité** dans le réseau de neurones, ce qui lui permet d'apprendre des représentations complexes. Sans elles, un réseau multicouche se réduirait à une simple transformation linéaire.

---

## Sigmoid

La fonction sigmoïde compresse toute valeur réelle dans l'intervalle $]0, 1[$, ce qui la rend naturelle pour modéliser des **probabilités**.

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

**Dérivée :**
$$\sigma'(x) = \sigma(x)\,(1 - \sigma(x))$$

| Propriété | Valeur |
|---|---|
| Plage de sortie | $(0,\ 1)$ |
| Monotone | Oui |
| Problème | Saturation aux extrêmes → gradient qui s'évanouit (*vanishing gradient*) |

---

## ReLU — Rectified Linear Unit

La ReLU est la fonction d'activation la plus utilisée dans les réseaux profonds. Elle est simple, efficace et atténue le problème de gradient évanescent pour les valeurs positives.

$$\text{ReLU}(x) = \max(0,\ x)$$

**Dérivée :**
$$\text{ReLU}'(x) = \begin{cases} 1 & \text{si } x > 0 \\ 0 & \text{si } x \leq 0 \end{cases}$$

| Propriété | Valeur |
|---|---|
| Plage de sortie | $[0,\ +\infty)$ |
| Monotone | Oui |
| Problème | *Dying ReLU* : neurones bloqués à 0 si toutes les entrées sont négatives |

---

## Tanh — Tangente hyperbolique

La tanh est une version **centrée en zéro** de la sigmoïde, avec une sortie dans $]-1, 1[$. Cette propriété accélère souvent la convergence par rapport à la sigmoïde.

$$\tanh(x) = \frac{e^{x} - e^{-x}}{e^{x} + e^{-x}}$$

Elle peut aussi s'exprimer en fonction de la sigmoïde :

$$\tanh(x) = 2\,\sigma(2x) - 1$$

**Dérivée :**
$$\tanh'(x) = 1 - \tanh^2(x)$$

| Propriété | Valeur |
|---|---|
| Plage de sortie | $(-1,\ 1)$ |
| Centrée en zéro | Oui |
| Problème | Saturation aux extrêmes → gradient qui s'évanouit (*vanishing gradient*) |

---

## Comparaison rapide

| Fonction | Plage | Centrée en 0 | Vanishing gradient |
|---|---|---|---|
| Sigmoid | $(0,\ 1)$ | Non | Oui |
| ReLU | $[0,\ +\infty)$ | Non | Non (pour $x>0$) |
| Tanh | $(-1,\ 1)$ | **Oui** | Oui |


In [2]:
def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def relu(x):
    return max(0, x)

def tanh(x):
    return math.tanh(x)

# Neurone

Un neurone artificiel est l'unité de base d'un réseau de neurones. Il s'inspire du neurone biologique : il reçoit plusieurs signaux en entrée, les **pondère**, les **somme**, ajoute un **biais**, puis applique une **fonction d'activation** pour produire sa sortie.

---

## Formule

Étant donné :
- $\mathbf{x} = (x_1, x_2, \dots, x_n)$ — le vecteur d'entrées
- $\mathbf{w} = (w_1, w_2, \dots, w_n)$ — le vecteur de poids associés
- $b$ — le biais
- $f$ — la fonction d'activation

Le neurone calcule :

$$z = \sum_{i=1}^{n} x_i \cdot w_i + b = \mathbf{w} \cdot \mathbf{x} + b$$

$$\hat{y} = f(z)$$

où $z$ est appelé la **pré-activation** (ou *logit*) et $\hat{y}$ la **sortie** du neurone.

---

## Rôle de chaque composant

| Composant | Rôle |
|---|---|
| $x_i$ | Signal d'entrée (feature) |
| $w_i$ | Importance accordée à l'entrée $x_i$ — appris lors de l'entraînement |
| $b$ | Décale la fonction d'activation — permet au neurone de s'activer même si toutes les entrées sont nulles |
| $f$ | Introduit la non-linéarité (sigmoid, ReLU, tanh…) |

---

## Illustration

```
x₁ ──w₁─┐
x₂ ──w₂─┤
         ├──► z = Σ xᵢwᵢ + b ──► f(z) ──► ŷ
x₃ ──w₃─┤
         ...
xₙ ──wₙ─┘
```

Le biais $b$ joue le même rôle que l'ordonnée à l'origine dans une équation linéaire : sans lui, la frontière de décision serait contrainte à passer par l'origine.


In [3]:
def neuron(inputs, weights, bias, activation_function=sigmoid):
    total = sum(i * w for i, w in zip(inputs, weights)) + bias
    return activation_function(total)

# Couche & Réseau

## Couche (*Layer*)

Une couche est un **ensemble de neurones en parallèle** qui reçoivent tous les mêmes entrées et produisent chacun une sortie indépendante.

Étant donné une couche de $m$ neurones, chacun avec son propre vecteur de poids $\mathbf{w}_j$ et biais $b_j$ :

$$\mathbf{a} = \begin{pmatrix} f\!\left(\mathbf{w}_1 \cdot \mathbf{x} + b_1\right) \\ f\!\left(\mathbf{w}_2 \cdot \mathbf{x} + b_2\right) \\ \vdots \\ f\!\left(\mathbf{w}_m \cdot \mathbf{x} + b_m\right) \end{pmatrix}$$

Soit de façon compacte, en notant $W$ la matrice des poids et $\mathbf{b}$ le vecteur des biais :

$$\mathbf{a} = f\!\left(W\,\mathbf{x} + \mathbf{b}\right)$$

où $f$ est appliquée **élément par élément** (*element-wise*).

```
        ┌─ neurone 1 ─► a₁
x ──────┼─ neurone 2 ─► a₂
        ├─ neurone 3 ─► a₃
        └─    ...
```

---

## Réseau (*Network*)

Un réseau est une **séquence de couches** où la sortie de chaque couche devient l'entrée de la suivante. On parle de réseau **feedforward** (propagation vers l'avant).

Pour un réseau de $L$ couches :

$$\mathbf{a}^{(0)} = \mathbf{x}$$

$$\mathbf{a}^{(\ell)} = f\!\left(W^{(\ell)}\,\mathbf{a}^{(\ell-1)} + \mathbf{b}^{(\ell)}\right) \quad \text{pour } \ell = 1, \dots, L$$

$$\hat{\mathbf{y}} = \mathbf{a}^{(L)}$$

```
Entrées       Couche 1      Couche 2      Sortie
  x  ──────► a⁽¹⁾ ────────► a⁽²⁾ ──────► ŷ
              (m₁ neurones)  (m₂ neurones)
```

| Symbole | Signification |
|---|---|
| $L$ | Nombre de couches |
| $\mathbf{a}^{(\ell)}$ | Activations de la couche $\ell$ |
| $W^{(\ell)}$ | Matrice des poids de la couche $\ell$ |
| $\mathbf{b}^{(\ell)}$ | Vecteur des biais de la couche $\ell$ |
| $\hat{\mathbf{y}}$ | Sortie finale du réseau (prédiction) |

> La puissance du réseau multicouche vient de la **composition** de transformations non-linéaires : chaque couche apprend une représentation de plus en plus abstraite des données.


In [4]:
def layer(inputs, weights, biases, activation_function=sigmoid):
    return [neuron(inputs, w, b, activation_function) for w, b in zip(weights, biases)]

def network(inputs, weights, biases, activation_function=sigmoid):
    for w, b in zip(weights, biases):
        inputs = layer(inputs, w, b, activation_function)
    return inputs

# Fonctions de perte (*Loss functions*)

La fonction de perte mesure l'**écart entre la prédiction $\hat{y}$ du réseau et la valeur réelle $y$**. C'est elle que l'on cherche à minimiser lors de l'entraînement. Son gradient indique dans quelle direction ajuster les poids.

---

## MSE — Mean Squared Error

Utilisée pour les problèmes de **régression**. Elle pénalise les grandes erreurs de façon quadratique.

$$\mathcal{L}_{\text{MSE}} = \frac{1}{n} \sum_{i=1}^{n} \left(\hat{y}_i - y_i\right)^2$$

**Gradient par rapport à $\hat{y}_i$ :**

$$\frac{\partial \mathcal{L}_{\text{MSE}}}{\partial \hat{y}_i} = \frac{2}{n}\left(\hat{y}_i - y_i\right)$$

| Propriété | Valeur |
|---|---|
| Usage typique | Régression |
| Sensibilité aux *outliers* | Élevée (erreur au carré) |
| Sortie toujours positive | Oui |

---

## Cross-Entropy — Entropie croisée binaire

Utilisée pour les problèmes de **classification binaire** (sortie dans $]0, 1[$, typiquement avec sigmoid). Elle mesure la divergence entre la distribution prédite et la distribution réelle.

$$\mathcal{L}_{\text{CE}} = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

- Quand $y_i = 1$ : la perte est $-\log(\hat{y}_i)$, qui tend vers $+\infty$ si $\hat{y}_i \to 0$
- Quand $y_i = 0$ : la perte est $-\log(1 - \hat{y}_i)$, qui tend vers $+\infty$ si $\hat{y}_i \to 1$

**Gradient par rapport à $\hat{y}_i$ :**

$$\frac{\partial \mathcal{L}_{\text{CE}}}{\partial \hat{y}_i} = \frac{1}{n}\left( -\frac{y_i}{\hat{y}_i} + \frac{1 - y_i}{1 - \hat{y}_i} \right)$$

| Propriété | Valeur |
|---|---|
| Usage typique | Classification binaire |
| Sensibilité aux *outliers* | Modérée (pénalité logarithmique) |
| Requiert | $\hat{y}_i \in\ ]0, 1[$ (évite $\log(0)$) |

---

## Rôle du gradient dans l'entraînement

Le gradient de la perte est le point de départ de la **rétropropagation** (*backpropagation*). Il indique, pour chaque prédiction, dans quel sens et avec quelle intensité corriger le réseau :

$$\Delta w \propto -\frac{\partial \mathcal{L}}{\partial w}$$

> Un gradient positif signifie que augmenter $\hat{y}_i$ augmente la perte → il faut diminuer la prédiction. Un gradient négatif indique l'inverse.

---

## Comparaison rapide

| Fonction | Usage | Gradient | Plage de perte |
|---|---|---|---|
| MSE | Régression | $\frac{2}{n}(\hat{y}_i - y_i)$ | $[0,\ +\infty)$ |
| Cross-Entropy | Classification binaire | $\frac{1}{n}\!\left(\frac{1-y_i}{1-\hat{y}_i} - \frac{y_i}{\hat{y}_i}\right)$ | $[0,\ +\infty)$ |


In [5]:
def loss_mse(predicted, actual):
    return sum((p - a) ** 2 for p, a in zip(predicted, actual)) / len(predicted)

def loss_mse_gradient(predicted, actual):
    return [(p - a) * 2 / len(predicted) for p, a in zip(predicted, actual)]

def loss_cross_entropy(predicted, actual):
    return -sum(a * math.log(p) + (1 - a) * math.log(1 - p) for p, a in zip(predicted, actual)) / len(predicted)

def loss_cross_entropy_gradient(predicted, actual):
    return [(-a / p + (1 - a) / (1 - p)) / len(predicted) for p, a in zip(predicted, actual)]